# Atividade semana 12 - pré processamento (titanic)

objetivo: tratar nulos, escalonar e codificar as variaveis categoricas usando um pipeline

### importando as libs

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

### carregando o dataset

In [2]:
df = pd.read_csv("https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv")
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [3]:
# vendo o tamanho e os tipos
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


In [5]:
# checando os nulos
df.isnull().sum()

,0
PassengerId,0
Survived,0
Pclass,0
Name,0
Sex,0
Age,177
SibSp,0
Parch,0
Ticket,0
Fare,0


### separando X e y


In [6]:
colunas_num = ['Age', 'SibSp', 'Parch']
colunas_cat = ['Sex', 'Embarked', 'Pclass']

X = df[colunas_num + colunas_cat]
y = df['Survived']

### treino e teste

In [7]:
X_treino, X_teste, y_treino, y_teste = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(X_treino.shape)
print(X_teste.shape)

(712, 6)
(179, 6)


### montando o pipeline


In [8]:
def montar_pipeline(imputacao='median', scaler=StandardScaler(), modelo=LogisticRegression(max_iter=1000)):

    num_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy=imputacao)),
        ('scaler', scaler)
    ])

    cat_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore'))
    ])

    preprocessador = ColumnTransformer(transformers=[
        ('num', num_transformer, colunas_num),
        ('cat', cat_transformer, colunas_cat)
    ])

    pipeline = Pipeline(steps=[
        ('preprocessador', preprocessador),
        ('classificador', modelo)
    ])

    return pipeline

### rodando o modelo base
(mediana + standardscaler + regressao logistica)

In [9]:
pipe = montar_pipeline()
pipe.fit(X_treino, y_treino)

pred = pipe.predict(X_teste)
acc_base = accuracy_score(y_teste, pred)

print("acuracia base:", acc_base)

acuracia base: 0.8044692737430168


### 1) mudando a estrategia de imputacao

In [10]:
for estrategia in ['median', 'mean', 'constant']:
    pipe_teste = montar_pipeline(imputacao=estrategia)
    pipe_teste.fit(X_treino, y_treino)
    acc = accuracy_score(y_teste, pipe_teste.predict(X_teste))
    print(estrategia, '->', acc)

median -> 0.8044692737430168
mean -> 0.7988826815642458
constant -> 0.7877094972067039


### 2) mudando o scaler

In [11]:
pipe_standard = montar_pipeline(scaler=StandardScaler())
pipe_standard.fit(X_treino, y_treino)
print("StandardScaler:", accuracy_score(y_teste, pipe_standard.predict(X_teste)))

pipe_minmax = montar_pipeline(scaler=MinMaxScaler())
pipe_minmax.fit(X_treino, y_treino)
print("MinMaxScaler:", accuracy_score(y_teste, pipe_minmax.predict(X_teste)))

StandardScaler: 0.8044692737430168
MinMaxScaler: 0.7932960893854749


### 3) mudando o modelo

In [12]:
pipe_lr = montar_pipeline(modelo=LogisticRegression(max_iter=1000))
pipe_lr.fit(X_treino, y_treino)
print("LogisticRegression:", accuracy_score(y_teste, pipe_lr.predict(X_teste)))

pipe_tree = montar_pipeline(modelo=DecisionTreeClassifier(random_state=42))
pipe_tree.fit(X_treino, y_treino)
print("DecisionTree:", accuracy_score(y_teste, pipe_tree.predict(X_teste)))

LogisticRegression: 0.8044692737430168
DecisionTree: 0.8044692737430168


### 4) Se nao escalonar


In [16]:
num_sem_scaler = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median'))
])

cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

prep_sem_scaler = ColumnTransformer(transformers=[
    ('num', num_sem_scaler, colunas_num),
    ('cat', cat_transformer, colunas_cat)
])

pipe_sem_scaler = Pipeline(steps=[
    ('preprocessador', prep_sem_scaler),
    ('classificador', LogisticRegression(max_iter=1000))
])

pipe_sem_scaler.fit(X_treino, y_treino)
acc_sem_scaler = accuracy_score(y_teste, pipe_sem_scaler.predict(X_teste))

print("com scaler:", acc_base)
print("sem scaler:", acc_sem_scaler)

com scaler: 0.8044692737430168
sem scaler: 0.8044692737430168


### 5) Se nao tratar os nulos

In [17]:
# tirando o imputer
cat_sem_imputer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

prep_sem_nulo = ColumnTransformer(transformers=[
    ('num', 'passthrough', colunas_num),
    ('cat', cat_sem_imputer, colunas_cat)
])

pipe_sem_nulo = Pipeline(steps=[
    ('preprocessador', prep_sem_nulo),
    ('classificador', LogisticRegression(max_iter=1000))
])

pipe_sem_nulo.fit(X_treino, y_treino)

ValueError: Input X contains NaN.
LogisticRegression does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values

deu erro mesmo: `ValueError: Input X contains NaN`

faz sentido, porque a coluna Age tem varios nulos e o modelo nao sabe lidar com isso sozinho, por isso que precisa do SimpleImputer antes.

## Respostas

**1 - Qual estratégia deu melhor resultado? a diferença é grande?**

Testando mediana, média e constante pra preencher os nulos da idade, a acurácia ficou bem parecida entre elas (a diferença foi pequena, tipo 1 ou 2%). Isso meio que faz sentido porque a idade não tem tantos outliers assim que façam a média ficar muito diferente da mediana. Olhando o scaler, standard e minmax também deram resultado bem próximo. E entre os modelos, a regressão logística e a árvore de decisão deram acurácias parecidas, mudando um pouco dependendo da rodada.

**2 - O que acontece se não escalonar os dados?**

Sem o StandardScaler a acurácia mudou pouco nesse caso, mas isso é mais por sorte do dataset. Em teoria, pra modelo como regressão logística que usa os valores numéricos pra calcular os coeficientes, não escalonar pode fazer uma variável "pesar mais" que a outra só pq a escala dela é maior (tipo Age vai de 0 a 80 e Parch só vai de 0 a 6). Já pra árvore de decisão não faz diferença nenhuma, porque ela so olha pra fazer cortes/divisões e não pra distancia entre os valores.

**3 - O que acontece se não tratar os nulos?**

Da erro na hora de treinar o modelo (ValueError: Input X contains NaN), o código nem roda. Então isso aqui não é uma escolha opcional tipo o scaler, é obrigatório fazer ou o modelo simplesmente não treina.